# ICU Image Encoder — DenseNet-121 + Grad-CAM
> **Metadata source** : BigQuery `physionet-data.mimic_cxr` — file paths, labels, study info  
> **Image source** : GCS `gs://mimic-cxr-jpg` — 570 GB of JPEG chest X-rays, streamed on demand  
> **Model** : DenseNet-121 pretrained on CheXpert + NIH + PadChest (torchxrayvision)  
> **Output** : 1024-dim feature vector + 18 pathology probs + Grad-CAM heatmaps per stay  
> **Explainability** : Grad-CAM region analysis + plain-English clinical interpretation


## Step 0 — Install Dependencies

In [8]:
#!pip install "Pillow==9.5.0" "scikit-image==0.21.0" --force-reinstall -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 101.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.6/317.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.0/244.0 kB 16.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.3 which is incompa

In [9]:
import subprocess, sys

# Pin Pillow FIRST — torchxrayvision breaks on Pillow 10.x
# The current Python 3.12 environment is incompatible with Pillow 9.x. Removing this pin to allow a compatible version to be installed.
# subprocess.run([sys.executable,'-m','pip','install','-q','Pillow==9.5.0'], check=True)

PACKAGES = [
    'torch',        # Corrected: Install torch
    'torchvision',  # Corrected: Install torchvision
    'torchxrayvision',
    'google-cloud-bigquery',
    'google-cloud-storage',
    'db-dtypes',
    'umap-learn',
    'tqdm',
    'opencv-python-headless',   # for CAM resizing
]
for pkg in PACKAGES:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg],
                       capture_output=True, text=True)
    print(f'  {[{'OK' if r.returncode==0 else 'FAIL'}]} {pkg}')
    if r.returncode != 0:
        print(f"Error installing {pkg}:\n{r.stderr}")

import torch, torchxrayvision as xrv
print(f'\nPyTorch        : {torch.__version__}')
print(f'torchxrayvision: {xrv.__version__}')
print(f'GPU available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device     : {torch.cuda.get_device_name(0)}')


  [{'OK'}] torch
  [{'OK'}] torchvision
  [{'OK'}] torchxrayvision
  [{'OK'}] google-cloud-bigquery
  [{'OK'}] google-cloud-storage
  [{'OK'}] db-dtypes
  [{'OK'}] umap-learn
  [{'OK'}] tqdm
  [{'OK'}] opencv-python-headless

PyTorch        : 2.10.0+cpu
torchxrayvision: 1.4.0
GPU available  : False


## Step 1 — Environment Setup

In [14]:
import warnings, logging, io, json
from pathlib import Path
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('CXR_Encoder')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import cv2
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import torchxrayvision as xrv
from google.cloud import bigquery, storage as gcs
from google.colab import drive, auth

drive.mount('/content/drive')
auth.authenticate_user()

# ── CHANGE THIS to your GCP project ID
GCP_PROJECT    = 'icuaiassistanttextencoder'
GCS_BUCKET     = 'mimic-cxr-jpg'

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
log.info('Device: %s', DEVICE)

BASE      = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/Parquet')
MODEL_DIR = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud/Models')
OUT_DIR   = Path('/content/drive/MyDrive/Multimoal_ICUAIAssistant_Claud')
CACHE_DIR = Path('/content/cxr_cache')
CACHE_DIR.mkdir(exist_ok=True)

CXR_EMB_OUT  = BASE / 'icu_cxr_embeddings.parquet'
CXR_RPT_PNG  = OUT_DIR / 'icu_cxr_encoder_report.png'
CAM_OUT_DIR  = OUT_DIR / 'gradcam_samples'
CAM_OUT_DIR.mkdir(exist_ok=True)

# ── Plot theme
DARK_BG='#0F172A'; CARD_BG='#1E293B'; TEXT='#F1F5F9'; GRID='#334155'
PAL = {'blue':'#2563EB','teal':'#0891B2','purple':'#7C3AED',
       'green':'#16A34A','orange':'#EA580C','red':'#DC2626','amber':'#CA8A04'}
TIER_COLORS = {'LOW':'#16A34A','MODERATE':'#CA8A04','HIGH':'#EA580C',
               'SEVERE':'#DC2626','CRITICAL':'#7C3AED'}
TIER_ORDER  = ['LOW','MODERATE','HIGH','SEVERE','CRITICAL']

plt.rcParams.update({
    'figure.facecolor':DARK_BG,'axes.facecolor':CARD_BG,'axes.edgecolor':GRID,
    'axes.labelcolor':TEXT,'axes.titlecolor':TEXT,'xtick.color':TEXT,
    'ytick.color':TEXT,'text.color':TEXT,'grid.color':GRID,'grid.linewidth':0.5,
    'axes.grid':True,'axes.titlesize':12,'axes.labelsize':10,
    'xtick.labelsize':8,'ytick.labelsize':8,'legend.fontsize':8,
})
log.info('Environment ready.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Load ICU Stay Cohort

In [12]:
static  = pd.read_parquet(BASE / 'icu_static_features.parquet')
enc_feats = pd.read_parquet(BASE / 'icu_encoder_features.parquet')
scores  = pd.read_parquet(BASE / 'icu_risk_scores.parquet')

cohort = static[['stay_id','subject_id','hadm_id','intime','outtime']].copy()
cohort['intime']  = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])
cohort = cohort.merge(enc_feats[['stay_id','split']], on='stay_id', how='left')
cohort['split'] = cohort['split'].fillna('train')

SUBJECT_IDS = cohort['subject_id'].dropna().astype(int).unique().tolist()
log.info('Cohort: %d stays | %d subjects', len(cohort), len(SUBJECT_IDS))


## Step 3 — Pull CXR Metadata from BigQuery
> BigQuery holds: file paths, study dates, view positions, CheXpert labels.  
> It does NOT hold images — those stream from GCS in Step 6.


In [15]:
bq  = bigquery.Client(project=GCP_PROJECT)
gcs_client = gcs.Client(project=GCP_PROJECT)
bucket     = gcs_client.bucket(GCS_BUCKET)

def bq_pull_cxr_metadata(subject_ids: list) -> pd.DataFrame:
    """
    Pull frontal CXR metadata from BigQuery.
    Joins record_list + dicom_metadata_string + chexpert labels.
    Batched to stay within BQ IN-clause limits.
    """
    BATCH = 5000
    dfs = []
    for i in range(0, len(subject_ids), BATCH):
        ids_str = ','.join(str(x) for x in subject_ids[i:i+BATCH])
        q = f"""
        SELECT
            r.subject_id,
            r.study_id,
            r.dicom_id,
            r.path,
            CAST(m.StudyDate AS STRING)  AS study_date,
            m.ViewPosition,
            -- CheXpert labels from the labels table
            c.Atelectasis,
            c.Cardiomegaly,
            c.Consolidation,
            c.Edema,
            c.`Enlarged Cardiomediastinum`,
            c.Fracture,
            c.`Lung Lesion`,
            c.`Lung Opacity`,
            c.`No Finding`,
            c.`Pleural Effusion`,
            c.`Pleural Other`,
            c.Pneumonia,
            c.Pneumothorax,
            c.`Support Devices`
        FROM `physionet-data.mimic_cxr.record_list` r
        LEFT JOIN `physionet-data.mimic_cxr.dicom_metadata_string` m
            ON r.dicom_id = m.SOPInstanceUID
        LEFT JOIN `physionet-data.mimic_cxr_jpg.chexpert` c
            ON r.study_id = c.study_id
        WHERE r.subject_id IN ({ids_str})
          AND m.ViewPosition IN ('PA','AP')
        """
        df = bq.query(q).to_dataframe(progress_bar_type='tqdm')
        dfs.append(df)
        log.info('BQ batch %d/%d → %d rows',
                 i//BATCH+1, -(-len(subject_ids)//BATCH), len(df))
    return pd.concat(dfs, ignore_index=True)


log.info('Pulling CXR metadata from BigQuery...')
cxr_meta = bq_pull_cxr_metadata(SUBJECT_IDS)
log.info('Raw CXR metadata: %d rows', len(cxr_meta))

# ── Parse study date
cxr_meta['study_dt'] = pd.to_datetime(
    cxr_meta['study_date'], format='%Y%m%d', errors='coerce'
)
log.info('View distribution:\n%s',
         cxr_meta['ViewPosition'].value_counts().to_string())


Query is running:   0%|          |

Forbidden: 403 Access Denied: Table physionet-data:mimic_cxr.dicom_metadata_string: User does not have permission to query table physionet-data:mimic_cxr.dicom_metadata_string, or perhaps it does not exist.; reason: accessDenied, message: Access Denied: Table physionet-data:mimic_cxr.dicom_metadata_string: User does not have permission to query table physionet-data:mimic_cxr.dicom_metadata_string, or perhaps it does not exist.

Location: US
Job ID: 5951693c-d570-4c94-8e46-32d1d45de75a


## Step 4 — Temporal Filter & Best CXR Selection
> Keep only CXRs within 48h before ICU admission or during the stay.  
> One CXR per stay — the most recent within the window.


In [ ]:
cxr_linked = cxr_meta.merge(
    cohort[['subject_id','stay_id','intime','outtime','split']],
    on='subject_id', how='inner'
)

# Temporal window: 48h before ICU admission → end of ICU stay
cxr_linked['window_start'] = cxr_linked['intime'] - pd.Timedelta(hours=48)
cxr_linked['window_end']   = cxr_linked['outtime']

cxr_in_window = cxr_linked[
    (cxr_linked['study_dt'] >= cxr_linked['window_start']) &
    (cxr_linked['study_dt'] <= cxr_linked['window_end'])
].copy()

# One CXR per stay: most recent
cxr_selected = (
    cxr_in_window
    .sort_values(['stay_id','study_dt'], ascending=[True, False])
    .drop_duplicates(subset='stay_id', keep='first')
    .reset_index(drop=True)
)

cxr_selected['hours_from_icu'] = (
    (cxr_selected['study_dt'] - cxr_selected['intime'])
    .dt.total_seconds() / 3600
)

coverage = len(cxr_selected) / len(cohort) * 100
log.info('CXR coverage: %d / %d stays (%.1f%%)',
         len(cxr_selected), len(cohort), coverage)
log.info('View breakdown:\n%s',
         cxr_selected['ViewPosition'].value_counts().to_string())


## Step 5 — Load DenseNet-121
> Pretrained on CheXpert + NIH + PadChest simultaneously.  
> We hook the penultimate dense block to extract 1024-dim feature vectors.


In [ ]:
log.info('Loading DenseNet-121 (all-dataset weights)...')
model = xrv.models.DenseNet(weights='densenet121-res224-all')
model.eval()
model.to(DEVICE)

PATHOLOGY_LABELS = [str(p) for p in model.pathologies]
N_LABELS = len(PATHOLOGY_LABELS)
log.info('Pathology labels (%d): %s', N_LABELS, PATHOLOGY_LABELS)

# Column names for the output parquet
FEAT_COLS = [f'cxr_feat_{i}' for i in range(1024)]
PROB_COLS = [
    'cxr_prob_' + p.replace(' ','_').replace('/','_')
    for p in PATHOLOGY_LABELS
]

# Save label metadata
cxr_meta_info = {
    'pathology_labels': PATHOLOGY_LABELS,
    'feat_cols': FEAT_COLS,
    'prob_cols': PROB_COLS,
    'feat_dim': 1024,
    'model': 'densenet121-res224-all',
}
with open(MODEL_DIR / 'cxr_meta.json','w') as f:
    json.dump(cxr_meta_info, f, indent=2)
log.info('CXR metadata saved.')


## Step 6 — Image Loading from GCS + Preprocessing
> Images are streamed from GCS on demand — no 570 GB download needed.  
> Preprocessed to 224×224, normalised to xrv expected range [-1024, 1024].


In [ ]:
def load_jpeg_from_gcs(path: str) -> np.ndarray | None:
    """
    Stream a JPEG from GCS, convert to xrv-normalised numpy array.
    path format: 'files/p10/p10000032/s50414267/02aa804e-bde0afdd.jpg'
    Returns: (1, H, W) float32 array in [-1024, 1024] or None on failure.
    """
    try:
        blob      = bucket.blob(path)
        img_bytes = blob.download_as_bytes(timeout=30)
        img_pil   = Image.open(io.BytesIO(img_bytes)).convert('L')  # grayscale
        img_arr   = np.array(img_pil, dtype=np.float32)
        img_norm  = xrv.datasets.normalize(img_arr, maxval=255, reshape=True)
        return img_norm   # shape (1, H, W)
    except Exception as e:
        log.warning('GCS load failed for %s: %s', path, e)
        return None


def preprocess_image(img_arr: np.ndarray) -> torch.Tensor:
    """
    Apply xrv standard transforms and return (1,1,224,224) tensor.
    """
    transforms = xrv.datasets.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(224),
    ])
    img_t = transforms(img_arr)           # (1,224,224)
    return torch.tensor(img_t, dtype=torch.float32).unsqueeze(0)  # (1,1,224,224)


## Step 7 — Feature Extraction (Batched)
> 1024-dim features from denseblock4 + 18 pathology probabilities per image.


In [ ]:
@torch.no_grad()
def extract_features_batch(
    img_tensors: list[torch.Tensor]
) -> tuple[np.ndarray, np.ndarray]:
    """
    Args  : list of (1,1,224,224) tensors
    Returns:
      features   : (N, 1024) float32
      label_probs: (N, N_LABELS) float32
    """
    batch = torch.cat(img_tensors, dim=0).to(DEVICE)  # (N,1,224,224)

    feat_list = []
    def _hook(module, inp, out):
        # Global average pool of denseblock4 output (N, 1024, h, w) → (N, 1024)
        feat_list.append(out.mean(dim=[2,3]).cpu().numpy())

    hook = model.features.denseblock4.register_forward_hook(_hook)
    logits = model(batch)                              # (N, N_LABELS)
    hook.remove()

    probs    = torch.sigmoid(logits).cpu().numpy()    # (N, N_LABELS)
    features = feat_list[0]                           # (N, 1024)
    return features, probs


# ── Main encoding loop
BATCH_SIZE  = 8     # images per forward pass (reduce to 4 if OOM)
all_rows    = []
failed_stays = []

paths    = cxr_selected['path'].tolist()
stay_ids = cxr_selected['stay_id'].tolist()
splits   = cxr_selected['split'].tolist()

log.info('Encoding %d CXR images (batch=%d)...', len(paths), BATCH_SIZE)

i = 0
with tqdm(total=len(paths), desc='CXR encoding', unit='img') as pbar:
    while i < len(paths):
        b_paths    = paths[i:i+BATCH_SIZE]
        b_stay_ids = stay_ids[i:i+BATCH_SIZE]
        b_splits   = splits[i:i+BATCH_SIZE]

        # Load images
        loaded = [(sid, sp, load_jpeg_from_gcs(p))
                  for sid, sp, p in zip(b_stay_ids, b_splits, b_paths)]

        valid = [(sid, sp, img) for sid, sp, img in loaded if img is not None]
        failed_stays += [sid for sid, sp, img in loaded if img is None]

        if valid:
            sids   = [v[0] for v in valid]
            sps    = [v[1] for v in valid]
            tensors = [preprocess_image(v[2]) for v in valid]

            feats, probs = extract_features_batch(tensors)

            for j, (sid, sp) in enumerate(zip(sids, sps)):
                row = {'stay_id': sid, 'split': sp, 'cxr_available': 1}
                for k, col in enumerate(FEAT_COLS):
                    row[col] = float(feats[j, k])
                for k, col in enumerate(PROB_COLS):
                    row[col] = float(probs[j, k])
                all_rows.append(row)

        i += BATCH_SIZE
        pbar.update(len(b_paths))

log.info('Encoded: %d | Failed: %d', len(all_rows), len(failed_stays))

cxr_emb = pd.DataFrame(all_rows)

# Full cohort — stays without CXR get zero vector + flag=0
cxr_full = cohort[['stay_id','split']].merge(
    cxr_emb, on=['stay_id','split'], how='left'
)
cxr_full['cxr_available'] = cxr_full['cxr_available'].fillna(0).astype(int)
cxr_full[FEAT_COLS + PROB_COLS] = cxr_full[FEAT_COLS + PROB_COLS].fillna(0.0)

log.info('Final CXR embedding table: %s', cxr_full.shape)


## Step 8 — Save CXR Embeddings

In [ ]:
cxr_full.to_parquet(CXR_EMB_OUT, index=False,
                    engine='pyarrow', compression='snappy')
size_mb = CXR_EMB_OUT.stat().st_size / 1e6
log.info('Saved: %s | %.1f MB | shape: %s', CXR_EMB_OUT, size_mb, cxr_full.shape)
print(f'Columns: stay_id, split, cxr_available + {len(FEAT_COLS)} feat + {len(PROB_COLS)} prob')


## Step 9 — Grad-CAM Engine
> Hooks into DenseNet's final dense block.  
> Computes class-specific activation map → resized to original image dimensions.


In [ ]:
class GradCAM:
    """
    Class-discriminative Grad-CAM for DenseNet-121 (torchxrayvision).
    Hooks into denseblock4 — the last convolutional feature map before
    global average pooling and the classification head.
    """

    def __init__(self, densenet_model):
        self.model       = densenet_model
        self._activations = None
        self._gradients   = None
        self._fwd_hook = densenet_model.features.denseblock4\
            .register_forward_hook(self._save_activations)
        self._bwd_hook = densenet_model.features.denseblock4\
            .register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self._activations = out

    def _save_gradients(self, module, grad_in, grad_out):
        self._gradients = grad_out[0]

    def remove_hooks(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()

    def generate(self,
                 img_tensor: torch.Tensor,
                 class_idx: int) -> np.ndarray:
        """
        img_tensor : (1,1,224,224) on DEVICE
        class_idx  : index in PATHOLOGY_LABELS to explain
        Returns    : (224,224) float32 heatmap normalised [0,1]
        """
        self.model.zero_grad()
        img_tensor = img_tensor.requires_grad_(True)

        logits = self.model(img_tensor)           # (1, N_LABELS)
        score  = logits[0, class_idx]
        score.backward()

        # Global average pool of gradients: (1, C, H, W) → (1, C, 1, 1)
        weights = self._gradients.mean(dim=[2,3], keepdim=True)

        # Weighted sum of activation maps
        cam = (weights * self._activations).sum(dim=1, keepdim=True)  # (1,1,H,W)
        cam = F.relu(cam)                         # keep positive influence only

        # Upsample to 224×224
        cam_up = F.interpolate(
            cam, size=(224,224),
            mode='bilinear', align_corners=False
        ).squeeze().detach().cpu().numpy()

        # Normalise 0→1
        cam_up = np.maximum(cam_up, 0)
        if cam_up.max() > 0:
            cam_up /= cam_up.max()
        return cam_up.astype(np.float32)


grad_cam = GradCAM(model)
log.info('Grad-CAM hooks registered on denseblock4.')


## Step 10 — Anatomical Region Map & Plain-Language Lookup
> The 224×224 CAM is divided into a 3×3 anatomical grid.  
> Each region is named and given a clinical meaning a non-radiologist can understand.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  3 × 3 CXR anatomical grid (row, col) → (region name, plain meaning)
#  Based on standard PA/AP chest X-ray anatomy
# ─────────────────────────────────────────────────────────────────────────
REGION_MAP = {
    (0,0): ('Upper-left lung',
            'Left upper lobe — infections, masses, or collapse here'),
    (0,1): ('Upper-central / trachea',
            'Trachea and upper windpipe — check for shift or narrowing'),
    (0,2): ('Upper-right lung',
            'Right upper lobe — TB, pneumonia, and tumours appear here'),
    (1,0): ('Left mid-lung / hilum',
            'Left airways and blood vessels — enlarged nodes or consolidation'),
    (1,1): ('Heart and central chest',
            'Heart size and shape — enlarged heart suggests heart failure'),
    (1,2): ('Right mid-lung / hilum',
            'Right airways — consolidation or fluid common here'),
    (2,0): ('Left base',
            'Left lower lobe and angle — fluid, collapse, and infection'),
    (2,1): ('Lower-central / diaphragm',
            'Diaphragm contour — blunting suggests fluid underneath'),
    (2,2): ('Right base',
            'Right lower lobe — most common site for hospital-acquired pneumonia'),
}

# ─────────────────────────────────────────────────────────────────────────
#  Pathology → plain English
# ─────────────────────────────────────────────────────────────────────────
PATHOLOGY_PLAIN = {
    'Atelectasis':               'partial or complete lung collapse',
    'Cardiomegaly':              'enlarged heart — may indicate heart failure',
    'Consolidation':             'lung tissue filled with fluid or pus (pneumonia pattern)',
    'Edema':                     'fluid in the lung tissue — heart failure or ARDS pattern',
    'Enlarged Cardiomediastinum':'widened central chest — aortic or cardiac enlargement',
    'Fracture':                  'broken bone visible — rib or clavicle',
    'Lung Lesion':               'abnormal spot or mass in the lung',
    'Lung Opacity':              'haziness in the lung — infection, fluid, or blood',
    'No Finding':                'no obvious abnormality detected on this X-ray',
    'Pleural Effusion':          'fluid collecting around the lung',
    'Pleural Other':             'other abnormality of the lung lining',
    'Pneumonia':                 'lung infection — bacteria or virus',
    'Pneumothorax':              'air leak collapsing the lung — urgent',
    'Support Devices':           'tubes, lines, or devices inserted in the patient',
}

# Severity tier for each pathology (for colour coding)
PATHOLOGY_SEVERITY = {
    'Pneumothorax':              'critical',
    'Consolidation':             'high',
    'Edema':                     'high',
    'Lung Opacity':              'high',
    'Pneumonia':                 'high',
    'Pleural Effusion':          'moderate',
    'Atelectasis':               'moderate',
    'Cardiomegaly':              'moderate',
    'Enlarged Cardiomediastinum':'moderate',
    'Lung Lesion':               'moderate',
    'Fracture':                  'low',
    'Pleural Other':             'low',
    'Support Devices':           'info',
    'No Finding':                'normal',
}
SEVERITY_COLOR = {
    'critical': '#DC2626', 'high': '#EA580C',
    'moderate': '#CA8A04', 'low': '#2563EB',
    'info': '#0891B2',     'normal': '#16A34A',
}


def get_activated_regions(cam: np.ndarray,
                          threshold: float = 0.35) -> list[dict]:
    """Divide 224x224 CAM into 3x3 grid, return regions above threshold."""
    h, w  = cam.shape
    gh, gw = h//3, w//3
    regions = []
    for r in range(3):
        for c in range(3):
            cell = cam[r*gh:(r+1)*gh, c*gw:(c+1)*gw]
            act  = float(cell.mean())
            if act >= threshold:
                name, meaning = REGION_MAP[(r,c)]
                regions.append({
                    'region': name, 'meaning': meaning,
                    'activation': round(act, 3), 'grid': (r,c)
                })
    return sorted(regions, key=lambda x: x['activation'], reverse=True)


## Step 11 — Full Grad-CAM Visualisation + Plain-Language Report
> 5-panel layout per patient:  
> Original CXR · Grad-CAM overlay · Region grid · Pathology bar chart · Plain-language card


In [ ]:
def explain_cxr_full(
    stay_id: int,
    target_label: str | None = None,
    save: bool = True
) -> None:
    """
    Complete Grad-CAM explanation for one ICU patient's chest X-ray.
    Automatically selects the highest-probability pathology as Grad-CAM target
    unless target_label is specified.
    """
    # ── Get CXR path
    row = cxr_selected[cxr_selected['stay_id'] == stay_id]
    if len(row) == 0:
        print(f'No CXR found for stay {stay_id}')
        return
    path      = row['path'].iloc[0]
    view_pos  = row['ViewPosition'].iloc[0]
    study_dt  = row['study_dt'].iloc[0]

    # ── Load image from GCS
    img_arr = load_jpeg_from_gcs(path)
    if img_arr is None:
        print(f'Could not load image from GCS: {path}')
        return

    # ── Preprocess
    img_tensor = preprocess_image(img_arr).to(DEVICE)
    img_224    = img_tensor.squeeze().cpu().numpy()  # (224,224)

    # ── Get pathology probabilities for this stay
    emb_row = cxr_emb[cxr_emb['stay_id'] == stay_id]
    if len(emb_row) == 0:
        print(f'No embedding found for stay {stay_id}')
        return

    prob_dict = {
        PATHOLOGY_LABELS[k]: float(emb_row[PROB_COLS[k]].iloc[0])
        for k in range(N_LABELS)
    }

    # ── Auto-select target: highest-prob non-'No Finding' pathology
    if target_label is None:
        filtered = {k:v for k,v in prob_dict.items() if k != 'No Finding'}
        target_label = max(filtered, key=filtered.get)
    target_idx = PATHOLOGY_LABELS.index(target_label)

    # ── Generate Grad-CAM
    cam = grad_cam.generate(img_tensor, target_idx)

    # ── Region analysis
    top_regions = get_activated_regions(cam, threshold=0.30)

    # ── Patient risk info
    score_row = scores[scores['stay_id'] == stay_id]
    static_row = static[static['stay_id'] == stay_id]
    tier = score_row['criticality_tier'].iloc[0] if len(score_row) > 0 else 'UNKNOWN'
    mort = score_row['apache2_pred_mortality'].iloc[0] if len(score_row) > 0 else 0
    age  = static_row['age_icu'].iloc[0] if len(static_row) > 0 else '?'

    # ─────────────────────────────────────────────────────────────────────
    #  FIGURE: 5 panels
    # ─────────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(26, 10), facecolor=DARK_BG)
    fig.suptitle(
        f'Chest X-Ray Analysis  |  Stay {stay_id}  |  '
        f'Age {age}  |  Criticality: {tier}  |  '
        f'Mortality risk: {mort*100:.1f}%',
        fontsize=13, fontweight='bold', color=TEXT, y=1.01
    )

    gs_main = gridspec.GridSpec(
        1, 5, figure=fig,
        width_ratios=[1, 1, 1, 1.1, 1.5], wspace=0.06
    )

    # ── Panel 1: Original X-ray
    ax1 = fig.add_subplot(gs_main[0])
    ax1.imshow(img_224, cmap='bone', aspect='auto')
    ax1.set_title(f'Original CXR\n({view_pos})', color=TEXT, fontsize=10)
    ax1.axis('off')

    # ── Panel 2: Grad-CAM heatmap overlay
    ax2 = fig.add_subplot(gs_main[1])
    # Normalise image to [0,1] for display
    img_disp = (img_224 - img_224.min()) / (img_224.max() - img_224.min() + 1e-8)
    img_rgb  = np.stack([img_disp]*3, axis=-1)

    # Apply jet colormap to CAM
    cam_colored = cm.jet(cam)[:,:,:3]          # (224,224,3)
    overlay = 0.55 * img_rgb + 0.45 * cam_colored
    overlay = np.clip(overlay, 0, 1)

    ax2.imshow(overlay, aspect='auto')
    ax2.set_title(
        f'Grad-CAM: "{target_label}"\n'
        f'(prob: {prob_dict[target_label]*100:.1f}%)',
        color=TEXT, fontsize=10
    )
    ax2.axis('off')

    # Colourbar for CAM
    sm = cm.ScalarMappable(cmap='jet',
                           norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cb = plt.colorbar(sm, ax=ax2, fraction=0.04, pad=0.02)
    cb.set_label('Attention\nintensity', color=TEXT, fontsize=7)
    cb.ax.yaxis.set_tick_params(color=TEXT)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color=TEXT, fontsize=7)

    # ── Panel 3: Region grid overlay
    ax3 = fig.add_subplot(gs_main[2])
    ax3.imshow(img_disp, cmap='bone', aspect='auto')
    ax3.set_title('Anatomical regions\n(model focus zones)', color=TEXT, fontsize=10)
    ax3.axis('off')

    cell_size = 224 / 3
    for reg in top_regions:
        r, c = reg['grid']
        act  = reg['activation']
        alpha = 0.2 + act * 0.5
        color = '#DC2626' if act > 0.6 else '#EA580C' if act > 0.4 else '#CA8A04'
        rect = plt.Rectangle(
            (c * cell_size, r * cell_size), cell_size, cell_size,
            fill=True, facecolor=color, alpha=alpha,
            edgecolor='white', linewidth=1
        )
        ax3.add_patch(rect)
        ax3.text(
            c * cell_size + cell_size/2,
            r * cell_size + cell_size/2,
            f'{act*100:.0f}%',
            color='white', fontsize=8, fontweight='bold',
            ha='center', va='center'
        )

    # ── Panel 4: Pathology probability bar chart
    ax4 = fig.add_subplot(gs_main[3])
    ax4.set_facecolor(CARD_BG)
    sorted_probs = sorted(prob_dict.items(), key=lambda x: x[1], reverse=True)
    labels  = [p[0] for p in sorted_probs]
    values  = [p[1]*100 for p in sorted_probs]
    bar_colors = [
        SEVERITY_COLOR.get(PATHOLOGY_SEVERITY.get(l,'low'), '#888')
        for l in labels
    ]
    bars = ax4.barh(range(len(labels)), values,
                    color=bar_colors, edgecolor=DARK_BG, height=0.65)
    for bar, val, lbl in zip(bars, values, labels):
        ax4.text(min(val+0.5, 95), bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=7, color=TEXT)
        if lbl == target_label:
            bar.set_edgecolor('white')
            bar.set_linewidth(2)
    ax4.set_yticks(range(len(labels)))
    ax4.set_yticklabels(labels, fontsize=7)
    ax4.set_xlim(0, 105)
    ax4.set_xlabel('Probability (%)', color=TEXT, fontsize=8)
    ax4.set_title('Detected conditions\n(model probability)', color=TEXT, fontsize=10)
    ax4.spines['top'].set_visible(False)
    ax4.spines['right'].set_visible(False)

    # ── Panel 5: Plain-language explanation card
    ax5 = fig.add_subplot(gs_main[4])
    ax5.set_facecolor('#0F172A')
    ax5.axis('off')

    def write(y, txt, color=TEXT, size=9, bold=False, indent=0):
        ax5.text(0.04 + indent*0.04, y, txt,
                 color=color, fontsize=size,
                 fontweight='bold' if bold else 'normal',
                 transform=ax5.transAxes, va='top',
                 wrap=True)

    y = 0.97
    write(y, 'WHAT THIS X-RAY IS SHOWING', TEXT, 11, bold=True); y -= 0.07
    write(y, f'Study date : {study_dt.strftime("%Y-%m-%d") if pd.notna(study_dt) else "unknown"}',
          '#94A3B8', 8); y -= 0.05
    write(y, f'View       : {view_pos}',
          '#94A3B8', 8); y -= 0.07

    write(y, 'PRIMARY FINDING', '#F1F5F9', 10, bold=True); y -= 0.06
    sev_color = SEVERITY_COLOR.get(
        PATHOLOGY_SEVERITY.get(target_label,'low'),'#888')
    write(y, f'{target_label}  ({prob_dict[target_label]*100:.1f}% likely)',
          sev_color, 10, bold=True); y -= 0.05
    plain_meaning = PATHOLOGY_PLAIN.get(target_label, target_label)
    write(y, f'This means: {plain_meaning}', '#CBD5E1', 8, indent=1); y -= 0.08

    write(y, 'WHERE THE MODEL FOCUSED', '#F1F5F9', 10, bold=True); y -= 0.06
    for reg in top_regions[:4]:
        act_pct = reg['activation'] * 100
        col = '#DC2626' if act_pct>60 else '#EA580C' if act_pct>40 else '#CA8A04'
        write(y, f'{reg["region"]}  ({act_pct:.0f}% focus)', col, 9, bold=True)
        y -= 0.045
        write(y, reg['meaning'], '#94A3B8', 7.5, indent=1); y -= 0.055

    write(y, 'OTHER SIGNIFICANT FINDINGS', '#F1F5F9', 10, bold=True); y -= 0.06
    shown = 0
    for label, prob in sorted_probs:
        if label == target_label or prob < 0.15:
            continue
        sev = PATHOLOGY_SEVERITY.get(label,'low')
        if sev in ('critical','high','moderate'):
            col2 = SEVERITY_COLOR.get(sev,'#888')
            write(y, f'{label}: {prob*100:.1f}%  = '
                  f'{PATHOLOGY_PLAIN.get(label,label)}',
                  col2, 8); y -= 0.055
            shown += 1
            if shown >= 3 or y < 0.05:
                break

    if shown == 0:
        write(y, 'No other significant findings above threshold.', '#94A3B8', 8)

    plt.tight_layout()

    if save:
        out_path = CAM_OUT_DIR / f'gradcam_stay_{stay_id}_{target_label.replace(" ","_")}.png'
        plt.savefig(out_path, dpi=160, bbox_inches='tight', facecolor=DARK_BG)
        log.info('Saved: %s', out_path)

    plt.show()
    plt.close()

    # ── Print plain-text summary to console
    print('\n' + '='*65)
    print(f'  CXR EXPLANATION — Stay {stay_id}')
    print('='*65)
    print(f'  Primary finding : {target_label} ({prob_dict[target_label]*100:.1f}% probable)')
    print(f'  In plain terms  : {PATHOLOGY_PLAIN.get(target_label,target_label)}')
    print(f'\n  Where the model focused:')
    for reg in top_regions[:4]:
        print(f'    [{reg["activation"]*100:.0f}% focus]  '
              f'{reg["region"]}: {reg["meaning"]}')
    print(f'\n  All detected conditions:')
    for lbl, prob in sorted_probs:
        flag = ' <-- primary target' if lbl == target_label else ''
        severity_tag = PATHOLOGY_SEVERITY.get(lbl,'')
        if prob >= 0.10:
            print(f'    {prob*100:5.1f}%  {lbl}: '
                  f'{PATHOLOGY_PLAIN.get(lbl,lbl)}{flag}')
    print('='*65)


# ── Run on 3 sample stays from test set
test_stays_with_cxr = [
    sid for sid in cohort.loc[cohort['split']=='test','stay_id'].values
    if sid in cxr_emb['stay_id'].values
][:3]

for sid in test_stays_with_cxr:
    explain_cxr_full(sid)


## Step 12 — CXR Encoder Report Dashboard

In [ ]:
from umap import UMAP

fig = plt.figure(figsize=(24, 22), facecolor=DARK_BG)
fig.suptitle('DenseNet-121 CXR Encoder — Report Dashboard',
             fontsize=18, fontweight='bold', color=TEXT, y=0.99)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.52, wspace=0.38)
def sp(ax): ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# ── 1. UMAP of CXR features by criticality tier
ax1 = fig.add_subplot(gs[0, :2])
has_cxr = cxr_full[cxr_full['cxr_available']==1]
sample  = has_cxr.sample(min(3000, len(has_cxr)), random_state=42)
tier_map = scores.set_index('stay_id')['criticality_tier']
tiers    = sample['stay_id'].map(tier_map).fillna('LOW')
umap_2d  = UMAP(n_components=2, n_neighbors=30,
                min_dist=0.1, random_state=42)\
               .fit_transform(sample[FEAT_COLS].values)
for tier in TIER_ORDER:
    m = tiers == tier
    ax1.scatter(umap_2d[m,0], umap_2d[m,1],
                c=TIER_COLORS[tier], s=6, alpha=0.5,
                label=tier, edgecolors='none')
ax1.set_title('UMAP of CXR Features (coloured by criticality tier)')
ax1.set_xlabel('UMAP-1'); ax1.set_ylabel('UMAP-2')
ax1.legend(markerscale=3, framealpha=0.2, facecolor=CARD_BG); sp(ax1)

# ── 2. CXR coverage by split
ax2 = fig.add_subplot(gs[0, 2])
cov = cxr_full.groupby('split')['cxr_available'].agg(['sum','count'])
cov['pct'] = cov['sum'] / cov['count'] * 100
cov = cov.reindex(['train','val','test'])
b = ax2.bar(cov.index, cov['pct'],
            color=[PAL['blue'],PAL['teal'],PAL['purple']],
            edgecolor=DARK_BG, width=0.5)
for bar, val in zip(b, cov['pct']):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.5,
             f'{val:.1f}%', ha='center', fontsize=9, color=TEXT)
ax2.set_title('CXR Coverage by Split'); ax2.set_ylabel('% stays')
ax2.set_ylim(0,105); sp(ax2)

# ── 3. Pathology prevalence (prob > 0.5) ranked bar
ax3 = fig.add_subplot(gs[1, :])
prev = (
    has_cxr[PROB_COLS]
    .apply(lambda col: (col > 0.5).mean() * 100)
    .rename(index=dict(zip(PROB_COLS, PATHOLOGY_LABELS)))
    .sort_values(ascending=False)
)
bar_c = [SEVERITY_COLOR.get(PATHOLOGY_SEVERITY.get(l,'low'),'#888')
         for l in prev.index]
bars3 = ax3.bar(range(len(prev)), prev.values,
                color=bar_c, edgecolor=DARK_BG, width=0.7)
ax3.set_xticks(range(len(prev)))
ax3.set_xticklabels(prev.index, rotation=30, ha='right', fontsize=8)
for bar, val in zip(bars3, prev.values):
    ax3.text(bar.get_x()+bar.get_width()/2, val+0.2,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=7, color=TEXT)
ax3.set_title('Pathology Prevalence Among Stays with CXR (prob > 0.5)')
ax3.set_ylabel('% of stays'); sp(ax3)
legend_patches = [mpatches.Patch(color=v, label=k)
                  for k,v in SEVERITY_COLOR.items()]
ax3.legend(handles=legend_patches, framealpha=0.2, facecolor=CARD_BG, ncol=3)

# ── 4. Pathology heatmap by criticality tier
ax4 = fig.add_subplot(gs[2, :])
cxr_tier = cxr_full.merge(
    scores[['stay_id','criticality_tier']], on='stay_id', how='left'
)
hmap = (
    cxr_tier.groupby('criticality_tier')[PROB_COLS].mean() * 100
).reindex(TIER_ORDER)
im = ax4.imshow(hmap.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=40)
ax4.set_xticks(range(len(PROB_COLS)))
ax4.set_xticklabels(PATHOLOGY_LABELS, rotation=35, ha='right', fontsize=7)
ax4.set_yticks(range(len(TIER_ORDER)))
ax4.set_yticklabels(TIER_ORDER, fontsize=8)
for i in range(len(TIER_ORDER)):
    for j in range(len(PROB_COLS)):
        v = hmap.values[i,j]
        if not np.isnan(v):
            ax4.text(j,i,f'{v:.1f}',ha='center',va='center',
                     fontsize=6, color='white' if v>20 else TEXT)
plt.colorbar(im, ax=ax4, label='Mean probability (%)', shrink=0.5)
ax4.set_title('Mean Pathology Probabilities by Criticality Tier (%)')

# ── 5. CXR timing relative to ICU admission
ax5 = fig.add_subplot(gs[3, 0])
hours = cxr_selected['hours_from_icu'].clip(-48, 48)
ax5.hist(hours, bins=40, color=PAL['teal'], edgecolor=DARK_BG)
ax5.axvline(0, color='white', ls='--', lw=1.5, label='ICU admission')
ax5.set_title('CXR Timing\nvs ICU Admission')
ax5.set_xlabel('Hours (negative = before ICU)')
ax5.set_ylabel('Studies'); ax5.legend(framealpha=0.2); sp(ax5)

# ── 6. View position split
ax6 = fig.add_subplot(gs[3, 1])
vp = cxr_selected['ViewPosition'].value_counts()
ax6.pie(vp, labels=vp.index, autopct='%1.1f%%',
        colors=[PAL['blue'],PAL['teal']],
        wedgeprops=dict(width=0.5, edgecolor=DARK_BG, linewidth=2),
        textprops={'color':TEXT,'fontsize':9})
ax6.set_title('View Position')

# ── 7. Feature norm distribution by tier
ax7 = fig.add_subplot(gs[3, 2])
norms = np.linalg.norm(cxr_full[FEAT_COLS].values, axis=1)
cxr_full['feat_norm'] = norms
cxr_norm_tier = cxr_full.merge(
    scores[['stay_id','criticality_tier']], on='stay_id', how='left'
)
for tier in TIER_ORDER:
    d = cxr_norm_tier.loc[
        (cxr_norm_tier['criticality_tier']==tier) &
        (cxr_norm_tier['cxr_available']==1), 'feat_norm'
    ]
    if len(d) > 5:
        ax7.hist(d, bins=30, alpha=0.65,
                 color=TIER_COLORS[tier], label=tier, edgecolor='none')
ax7.set_title('Feature Vector Norm\nby Criticality Tier')
ax7.set_xlabel('L2 norm'); ax7.set_ylabel('Stays')
ax7.legend(framealpha=0.2, facecolor=CARD_BG); sp(ax7)

plt.savefig(CXR_RPT_PNG, dpi=180, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
log.info('Report saved: %s', CXR_RPT_PNG)


## Step 13 — Output Summary

In [ ]:
print('=== CXR IMAGE ENCODER OUTPUTS ===')
for path in [CXR_EMB_OUT, CXR_RPT_PNG,
             MODEL_DIR/'cxr_meta.json']:
    p = Path(path)
    exists = p.exists()
    size   = p.stat().st_size/1e6 if exists else 0
    print(f'  {"OK" if exists else "MISSING":6s} | {size:6.1f} MB | {p.name}')

print(f'\n=== COVERAGE ===')
print(f'  Total ICU stays     : {len(cohort):,}')
print(f'  CXR within window   : {len(cxr_selected):,} ({len(cxr_selected)/len(cohort)*100:.1f}%)')
print(f'  Successfully encoded: {cxr_full["cxr_available"].sum():,}')
print(f'  Failed (GCS error)  : {len(failed_stays):,}')

print(f'\n=== EMBEDDING DIMENSIONS ===')
print(f'  Feature vector : {len(FEAT_COLS)} dims (penultimate DenseNet layer)')
print(f'  Pathology probs: {len(PROB_COLS)} labels')
print(f'  Total columns  : {cxr_full.shape[1]}')

print(f'\n=== READY FOR FUSION LAYER ===')
print('  Next: ICUFusionLayer.ipynb')
print('  Inputs:')
print('    icu_encoder_features.parquet    (time-series XGBoost features)')
print('    icu_text_embeddings.parquet     (768-dim BERT per stay)')
print('    icu_cxr_embeddings.parquet      (1024-dim DenseNet per stay)')
print('    icu_encoder_predictions.parquet (risk score predictions)')
